<a href="https://colab.research.google.com/github/Harshada900/pyspark-practise/blob/main/17_Delta_Lake_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Configure Spark with Delta Lake JARs
builder = SparkSession.builder \
    .appName("DeltaLake") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
print(spark.version)

4.1.2


In [2]:
data = [("Raj",   "Engineering", 50000),
        ("Priya", "Marketing",   70000),
        ("Amit",  "Engineering", 60000),
        ("Sara",  "HR",          80000)]

columns = ["name", "department", "salary"]
df = spark.createDataFrame(data, columns)


In [3]:
#write as delta instead of parquet

df.write.format("delta").mode("overwrite").save("/content/delta/employees")

In [6]:
#!pip install pyspark delta-spark

  Using cached delta_spark-4.3.0-py3-none-any.whl.metadata (2.2 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Using cached delta_spark-4.3.0-py3-none-any.whl (53 kB)
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008642 sha256=d03417dc7b6738bdfdd751121e0c69b024dca69e8d6315290ade2f19117b31f7
  Stored in directory: /root/.cache/pip/wheels/f4/ca/ea/203f40b3e935bbf99bee851c2f4a87d22996ab8212d367ce58
Successfully built pyspark
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.1.2
    Uninstalling pyspark-4.1.2:
      Successfully uninstalled pyspark-4.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 4.1.1 which is incompatible.


## **Reading Delta Table**

In [7]:
df = spark.read.format("delta").load("/content/delta/employees")
df.show()

+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
|  Raj|Engineering| 50000|
|Priya|  Marketing| 70000|
| Amit|Engineering| 60000|
| Sara|         HR| 80000|
+-----+-----------+------+



## **Schema Enforcement**

In [8]:
#trying to write data with wrong schema

wrong_df = spark.createDataFrame([
    ("Karan", "Marketing", "sixty thousand")  # salary as string!
], ["name", "department", "salary"])

wrong_df.write.format("delta").mode("append").save("/content/delta/employees")

AnalysisException: [DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'salary' and 'salary'

## **Schema Evolution**

In [9]:
# Add new column — need to explicitly allow it
new_df = spark.createDataFrame([
    ("Karan", "Marketing", 45000, "Mumbai")
], ["name", "department", "salary", "city"])

In [10]:
new_df.write.format("delta").mode("append").option("mergeSchema", "true").save("/content/delta/employees") #mergeschema allows adding cols explicitely

In [11]:
spark.read.format("delta").load("/content/delta/employees").show() #old data gets null for col city!

+-----+-----------+------+------+
| name| department|salary|  city|
+-----+-----------+------+------+
|Karan|  Marketing| 45000|Mumbai|
|  Raj|Engineering| 50000|  NULL|
|Priya|  Marketing| 70000|  NULL|
| Amit|Engineering| 60000|  NULL|
| Sara|         HR| 80000|  NULL|
+-----+-----------+------+------+



## **Time Travel** - Reading old versions of data

In [12]:
# Read current version
spark.read.format("delta").load("/content/delta/employees").show()

+-----+-----------+------+------+
| name| department|salary|  city|
+-----+-----------+------+------+
|Karan|  Marketing| 45000|Mumbai|
|  Raj|Engineering| 50000|  NULL|
|Priya|  Marketing| 70000|  NULL|
| Amit|Engineering| 60000|  NULL|
| Sara|         HR| 80000|  NULL|
+-----+-----------+------+------+



In [13]:
#read version 0 - original data beofore any changes!

spark.read.format("delta").option("versionAsOf", 0).load("/content/delta/employees").show() #no city col!

+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
|  Raj|Engineering| 50000|
|Priya|  Marketing| 70000|
| Amit|Engineering| 60000|
| Sara|         HR| 80000|
+-----+-----------+------+



In [15]:
#read yesterdays data using timestampAsOf

spark.read.format("delta").option("timestampAsOf", "2026-06-26 15:03:44.566").load("/content/delta/employees").show() # few minutes old data

+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
|  Raj|Engineering| 50000|
|Priya|  Marketing| 70000|
| Amit|Engineering| 60000|
| Sara|         HR| 80000|
+-----+-----------+------+



## **Check Delta Table History**

In [16]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/content/delta/employees")
delta_table.history().show()

+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2026-06-26 15:12:...|  NULL|    NULL|    WRITE|{mode -> Append, ...|NULL|    NULL|     NULL|          0|  Serializable|         true|{numFiles -> 2, n...|        NULL|Apache-Spark/4.1....|
|      0|2026-06-26 15:03:...|  NULL|    NULL|    WRITE|{mode -> Overwrit...|NULL|    NULL|     NULL|       NULL|  Serializable|        false|{numFiles -> 2, n...|        NULL|Apache-Spark/4.1....|
+-------+-

# **Vacuum — Cleaning Old Files**

In [18]:
# Delete files older than 7 days (default retention)
#delta_table.vacuum()

In [19]:
# Delete files older than 1 day (custom retention)
#delta_table.vacuum(retentionHours=24)